In [72]:
from dotenv import load_dotenv

load_dotenv()

True

In [73]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
"gemini-2.5-flash", # también: "gemini-1.5-pro", "gemini-1.5-flash", etc.
model_provider="google_genai",
temperature=0,
)
response = llm.invoke('Hola como estás')
response.text


'¡Hola! Estoy muy bien, gracias. ¿Y tú, cómo estás?'

In [74]:
response = llm.invoke('Qué clima hace en bogota?')
response.text

'El clima en Bogotá es generalmente **frío a templado**, con una característica principal: su **gran altitud** (aproximadamente 2640 metros sobre el nivel del mar).\n\nAquí te detallo sus principales características:\n\n1.  **Temperaturas:**\n    *   Las temperaturas promedio diurnas suelen oscilar entre los **18°C y 22°C**.\n    *   Por las noches y madrugadas, descienden considerablemente, pudiendo estar entre los **7°C y 12°C**.\n    *   Rara vez hace calor intenso y tampoco suele nevar (aunque en los páramos cercanos sí puede haber heladas).\n    *   Debido a su cercanía al ecuador, no tiene las cuatro estaciones marcadas como en otras latitudes, sino que las temperaturas son relativamente constantes a lo largo del año.\n\n2.  **Lluvias:**\n    *   Es una ciudad con **lluvias frecuentes**. No es raro que llueva en cualquier momento del día, incluso si antes estaba soleado.\n    *   Hay dos temporadas de lluvia más intensas (marzo-mayo y septiembre-noviembre) y dos temporadas "secas

In [75]:
system_prompt = """
Eres un agente de ventas para encontrar productos.
Y tus productos son:
    - Computadoras
    - Televisores
    - Refrigeradores
"""
messages = [
    ('system',system_prompt),
    ('user','Dime los productos que ofreces en la tienda?')
]
response = llm.invoke(messages)
response.text

'¡Claro! En nuestra tienda, ofrecemos una excelente selección de:\n\n*   **Computadoras**\n*   **Televisores**\n*   **Refrigeradores**\n\n¿Hay alguno en particular que te interese o te gustaría saber más sobre alguno de ellos?'

In [76]:
from langchain_core.tools import tool
import requests 

@tool("get_products",description="Get the products that the store offers filter by price") #Se agregan contextos a la tool para que sepa que hacer
def get_products(price: int):
    #Connect with API
    products = [
        {'name':'Computadoras','price':1000},
        {'name':'Mouse','price':100},
        {'name':'Teclado','price':200},
        {'name':'Audifonos','price':300},
    ]
    return "".join(f"{product['name']} - {product['price']}" for product in products)

In [77]:
get_products.invoke({'price':300})

'Computadoras - 1000Mouse - 100Teclado - 200Audifonos - 300'

In [87]:
from langchain_core.tools import tool
import requests 

@tool("get_products",description="Get the products that the store offers filter by price") #Se agregan contextos a la tool para que sepa que hacer
def get_products(price: int):
    url = 'https://api.escuelajs.co/api/v1/products'
    response = requests.get(url)
    products = response.json()
    return "".join(f"{product['title']} - {product['price']}" for product in products)

In [79]:
get_products.invoke({'price':300})

'Vibrant Pink Classic Sneakers - 86Futuristic Chic High-Heel Boots - 36Elegant Patent Leather Peep-Toe Pumps with Gold-Tone Heel - 54Elegant Purple Leather Loafers - 17Sleek Futuristic Electric Bicycle - 22Sleek All-Terrain Go-Kart - 37Radiant Citrus Eau de Parfum - 73Sleek Olive Green Hardshell Carry-On Luggage - 48Chic Transparent Fashion Handbag - 61Elegant Glass Tumbler Set - 50New Product - 10Laboris id similiqu - 92Consectetur corrupti - 76New Product02 - 10New Product 1764304119678 - 10New Product 1764305018717 - 10New Product 1764305052425 - 10New Product 1764305063550 - 10New Product 1764305195080 - 10New Product 1764305399598 - 10New Product 1764305721442 - 10New Product 1764305759191 - 10New Product 1764305775273 - 10New Product 1764305790222 - 10Earphone - 45New Product 1764308915828 - 10New Product 1764308943197 - 10New Product 1764309008096 - 10New Product 1764309022961 - 10New Product 1764309088944 - 10New Product 1764309262002 - 10New Product 1764309275889 - 10New Produ

In [80]:
@tool("login_ccg",description="Create a session web to web page") #Se agregan contextos a la tool para que sepa que hacer
def login_ccg(user: str, contr: str):
    url = '    data = {
        'user':user,
        'password':contr
    }
    response = requests.post(url, json=data)
    return response.json()


In [81]:
login_ccg.invoke({
    'user':'usuario@correo.com',
    'contr':'contraseña'})

{'jsonrpc': '2.0',
 'id': None,
 'result': {'status': 'Error',
  'message': 'Es necesario ingresar el usuario',
  'code': 400}}

In [88]:
@tool("get_weather",description="Get the weather of a city") #Se agregan contextos a la tool para que sepa que hacer
def get_weather(city: str):
    url = f'https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1'
    response = requests.get(url)
    city_json = response.json()
    latitude = city_json['results'][0]['latitude']
    longitude = city_json['results'][0]['longitude']
    url = f'https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true'
    response = requests.get(url)
    data = response.json()
    response = f"The weather in {city} is {data['current_weather']['temperature']}C with {data['current_weather']['windspeed']}km/h of wind"
    return response

In [83]:
get_weather.invoke({
    'city':'bogota'
})

'The weather in bogota is 13.8C with 1.5km/h of wind'

In [101]:
system_prompt = """
Eres un asistente de ventas que ayuda a los clientes a encontrar los productos que necesitan y dar el clima de una ciudad
Tus tools son:
- get_products: para obtener los productos que ofreces en la tienda
- get_weather: para obtener el clima de la ciudad
"""

messages = [
    ('system',system_prompt),
    ('user','Dime los productos que ofrece la tienda con un precio menor a 50')
]

llm_with_tools = llm.bind_tools([get_products,get_weather])
response = llm.invoke(messages)
#response.pretty_print()
response.tool_calls

KeyboardInterrupt: 

In [99]:
messages = [
    ('system',system_prompt),
    ('user','Dime el clima en bogota')
]

llm_with_tools = llm.bind_tools([get_products,get_weather])
response = llm.invoke(messages)
response.pretty_print()
response.tool_calls

================================== Ai Message ==================================

`get_weather`({"city": "bogota"})


[]

In [96]:
messages = [
    ('system',system_prompt),
    ('user','Hola como estas?')
]
llm_with_tools = llm.bind_tools([get_products,get_weather])
response = llm.invoke(messages)
response.pretty_print()
response.tool_calls

================================== Ai Message ==================================

¡Hola! Estoy muy bien, gracias por preguntar. Soy tu asistente de ventas y estoy aquí para ayudarte a encontrar lo que necesites.

¿En qué puedo ayudarte hoy? ¿Estás buscando algún producto en particular o te gustaría saber el clima de alguna ciudad?


[]